# RAFT QLoRA Fine-tune — Llama 3.1 8B Instruct
rank=8, alpha=16, lr=2e-5, 1 epoch, max_seq=3072, 4-bit NF4


In [ ]:
import subprocess, sys

# Pin torch to its current version before installing trl/peft.
# Without this, pip upgrades torch 2.10.0+cu128 → 2.12.1+cu130, which
# drops sm_60/P100 and breaks the pre-installed bitsandbytes.
res = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                     capture_output=True, text=True)
torch_ver_full = res.stdout.strip()   # e.g. '2.10.0+cu128'
torch_ver_base = torch_ver_full.split('+')[0]  # '2.10.0'
print(f'Pinning torch to: {torch_ver_base}')

# Include bitsandbytes: newer transformers (installed via trl/peft) requires >=0.46.1.
# On T4+cu128 this installs fine; torch pin prevents the torch upgrade that broke it on P100.
subprocess.run(['pip', 'install', '-q', 'trl', 'peft', 'accelerate', 'bitsandbytes',
                f'torch=={torch_ver_base}.*'], check=False)
subprocess.run(['pip', 'uninstall', '-y', 'torchao'], capture_output=True)

import torch
print(f'torch after install: {torch.__version__}')

In [ ]:
import torch, sys, subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap',
                           '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(f'GPU: {gpu_info}')
if gpu_info:
    cc = gpu_info.split(',')[-1].strip()
    sm = int(cc.replace('.', ''))  # '7.5' -> 75
    if sm < 70:
        raise RuntimeError(
            f'P100/sm_{sm} is not supported by PyTorch 2.3+. '
            'Go to the Kaggle notebook UI → Edit → Accelerator → select "GPU T4 x2", '
            'then re-run via kaggle kernels push.'
        )
    print(f'GPU OK: sm_{sm} is supported. Proceeding.')
else:
    sm = 0
    print('WARNING: no GPU detected, running on CPU')

In [ ]:
import os, json, torch, subprocess
from pathlib import Path

os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'
# Reduce allocator fragmentation — helps when GPU has <200 MiB free but needs 112 MiB
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

MODEL      = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
OUTPUT_DIR = Path('/kaggle/working/raft_llama31_8b')
RANK       = 8
ALPHA      = 16
LR         = 2e-5
# Data samples are ~832 tokens (3328 chars / ~4 chars per token).
# Capping at 1024 instead of 3072 prevents ~9 GB of attention-score tensors
# per forward pass that OOM GPU 1 when standard (non-flash) attention is used.
MAX_TOKENS = 1024
SEED       = 42
HF_TOKEN   = os.environ['HF_TOKEN']
RESPONSE_TEMPLATE = '<|start_header_id|>assistant<|end_header_id|>\n\n'

r = subprocess.run(['find', '/kaggle/input', '-name', 'train.jsonl'],
                   capture_output=True, text=True)
found = r.stdout.strip()
if found:
    DATA_DIR = Path(found).parent
    print(f'Found dataset at: {DATA_DIR}')
else:
    DATA_DIR = Path('/kaggle/input/raft-supply-chain')
    print('WARNING: train.jsonl not found, using default path')
    print('Input dirs:', list(Path('/kaggle/input').iterdir()))
print(f'model={MODEL}  rank={RANK}  lr={LR}  max_tokens={MAX_TOKENS}')

In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
import trl
print('trl version:', trl.__version__)


In [ ]:
class CompletionOnlyCollator:
    """Masks prompt tokens so loss is computed on assistant completions only."""
    def __init__(self, tokenizer, response_template):
        self.tokenizer = tokenizer
        self.template_ids = tokenizer.encode(response_template, add_special_tokens=False)

    def __call__(self, features):
        texts = [f['text'] for f in features]
        enc = self.tokenizer(
            texts, padding=True, truncation=True,
            max_length=MAX_TOKENS, return_tensors='pt'
        )
        labels = enc['input_ids'].clone()
        tpl = self.template_ids
        for i, seq in enumerate(labels):
            seq_list = seq.tolist()
            found = False
            for j in range(len(seq_list) - len(tpl) + 1):
                if seq_list[j:j+len(tpl)] == tpl:
                    labels[i, :j+len(tpl)] = -100
                    found = True
                    break
            if not found:
                labels[i] = -100  # no template found: skip this example
        labels[enc['input_ids'] == self.tokenizer.pad_token_id] = -100
        enc['labels'] = labels
        return enc

print('CompletionOnlyCollator defined')


In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

train_records = load_jsonl(DATA_DIR / 'train.jsonl')
eval_records  = load_jsonl(DATA_DIR / 'test.jsonl')
print(f'train={len(train_records)}  eval={len(eval_records)}')


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

def apply_template(record):
    return {'text': tokenizer.apply_chat_template(
        record['messages'], tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_records).map(apply_template)
eval_ds  = Dataset.from_list(eval_records).map(apply_template)
print('sample length (chars):', len(train_ds[0]['text']))
data_collator = CompletionOnlyCollator(tokenizer, RESPONSE_TEMPLATE)
print('collator ready')


In [ ]:
print('Entering cell: model loading')
import torch, gc

gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
if gpu:
    sm = gpu.major * 10 + gpu.minor
    print(f'GPU: {gpu.name}  VRAM: {gpu.total_memory/1e9:.1f}GB  sm_{sm}')
else:
    print('No GPU detected')
    sm = 0

n_gpus = torch.cuda.device_count()
USE_4BIT = sm >= 70
USE_8BIT = 60 <= sm < 70
print(f'USE_4BIT={USE_4BIT}  n_gpus={n_gpus}')

# device_map='auto' is required: transformers 4.45+ materialises the full FP16
# checkpoint (~16GB) before quantising. With device_map={'':0} that exceeds
# a single T4 (14.56GiB); with 'auto' it is split across both T4s (32GB total).
# Training uses the base Trainer (not SFTTrainer) to bypass TRL 1.7.1's
# _chunked_ce_forward which does `hidden[valid]` without moving `valid` to
# hidden's device, causing RuntimeError on multi-GPU layouts.
try:
    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=bnb_config,
            device_map='auto', token=HF_TOKEN,
        )
    elif USE_8BIT:
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=bnb_config,
            device_map='auto', token=HF_TOKEN,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            MODEL, device_map='auto', torch_dtype=torch.float16, token=HF_TOKEN,
        )
    print('from_pretrained OK')
    for i in range(torch.cuda.device_count()):
        print(f'  CUDA:{i}  {torch.cuda.memory_allocated(i)/1e9:.2f}GB allocated')
except Exception as e:
    print(f'from_pretrained FAILED: {type(e).__name__}: {e}')
    raise

try:
    if USE_4BIT or USE_8BIT:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={'use_reentrant': False},
        )
    else:
        model.config.use_cache = False
        model.enable_input_require_grads()
    print('model prepared OK')
except Exception as e:
    print(f'prepare_model FAILED: {type(e).__name__}: {e}')
    raise

print('model loaded and prepared')

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=RANK, lora_alpha=ALPHA,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', inference_mode=False,
)
# Wrap model with PEFT here so base Trainer sees an already-wrapped model.
# SFTTrainer used to do this internally, but we bypass SFTTrainer to avoid
# TRL 1.7.1's _chunked_cross_entropy_loss cross-device bug on device_map='auto'.
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=1,
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.05,
    bf16=(sm >= 80), fp16=(60 <= sm < 80),
    logging_steps=10,
    # Disable mid-training evaluation: eval forward pass holds all 32 layers of
    # activations simultaneously (~3.9 GB on GPU 1) because gradient checkpointing
    # is not applied during eval. GPU 1 only has ~3.1 GB free after model +
    # optimizer states, causing OOM at every eval_steps=100 checkpoint.
    # Adapter quality is assessed post-hoc by 23_eval_baseline_suite.py.
    eval_strategy='no',
    save_strategy='epoch',
    save_total_limit=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    report_to='none', seed=SEED, dataloader_pin_memory=False,
    remove_unused_columns=False,
)
trainer = Trainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    data_collator=data_collator,
)
trainer.train()


In [ ]:
adapter_path = OUTPUT_DIR / 'adapter'
model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))
with open(OUTPUT_DIR / 'train_config.json', 'w') as f:
    json.dump({'model': MODEL, 'rank': RANK, 'alpha': ALPHA, 'lr': LR,
               'epochs': 1, 'seed': SEED, 'max_tokens': MAX_TOKENS,
               'n_train': len(train_records), 'n_eval': len(eval_records),
               'response_template': RESPONSE_TEMPLATE}, f, indent=2)
print(f'adapter saved -> {adapter_path}')
print('Done. Download /kaggle/working/raft_llama31_8b/ from the Outputs tab.')
